In [ ]:
%load_ext autoreload
%autoreload 2

import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from vetuner.correction import (apply_correction, sample_correction_factors, select_samples,)
from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.gating import gate_samples
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.sensors import apply_sensor_model
from vetuner.surface_fit import (SURFACE_METHODS, aggregate_to_cells, blend_by_confidence, cell_sample_counts, fit_surface, gaussian_process_surface,)
from vetuner.ve_surface import (BASE_MAP_SURFACE,DEFAULT_MAP_AXIS,DEFAULT_RPM_AXIS,TRUE_SURFACE,ve_table,)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

train = generate_drive_cycle()
test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
base, truth = ve_table(params=BASE_MAP_SURFACE), ve_table(params=TRUE_SURFACE)

sim = simulate_engine(train, base)
log = apply_sensor_model(sim, seed=0)
kept = select_samples(log, gate_samples(log).accepted)

factors = sample_correction_factors(kept)
counts = cell_sample_counts(kept.rpm, kept.map_kpa)

In [ ]:
"""
Compare six scattered-data fitting methods on identical gated samples.

Each fits the same 9194 correction factors onto the 16 x 12 grid, and is
scored on table error against the known surface and on fuelling error over a
drive cycle never used to fit it.
"""

print(f"{len(kept)} gated samples, {(counts > 0).sum()} of {counts.size} cells visited\n")
print(f"{'method':18s} {'RMSE':>7} {'visited':>8} {'unvis':>8} {'max':>7} "
      f"{'AFR%':>7} {'ms':>7}")

results = {}
for method in SURFACE_METHODS:
    start = time.perf_counter()
    surface = fit_surface(method, factors, kept.rpm, kept.map_kpa)
    elapsed = (time.perf_counter() - start) * 1e3

    table = apply_correction(base, surface)
    results[method] = table

    t = evaluate_table(table, truth, counts)
    a = evaluate_afr(table, test)
    print(f"{method:18s} {t.rmse:7.2f} {t.rmse_visited:8.2f} {t.rmse_unvisited:8.2f} "
          f"{t.max_abs_error:7.2f} {a.mean_abs_pct:7.2f} {elapsed:7.1f}")

t = evaluate_table(base, truth, counts)
a = evaluate_afr(base, test)
print(f"\n{'starting map':18s} {t.rmse:7.2f} {t.rmse_visited:8.2f} "
      f"{t.rmse_unvisited:8.2f} {t.max_abs_error:7.2f} {a.mean_abs_pct:7.2f}")

In [ ]:
"""
Test whether the Gaussian process identifies where it lacks data.
Compares predicted standard deviation over cells the log reached against
cells it did not.
"""

surface, std = gaussian_process_surface(factors, kept.rpm, kept.map_kpa)

print(f"GP uncertainty, visited cells:   {std[counts > 0].mean():.4f}")
print(f"GP uncertainty, unvisited cells: {std[counts == 0].mean():.4f}")
print(f"ratio: {std[counts == 0].mean() / std[counts > 0].mean():.1f}x")

In [ ]:
"""
Sweep the confidence threshold used to fade the fit towards no correction.
Blending applies the full estimate where uncertainty is low and returns
towards a neutral factor of 1.0 where it is high. The threshold sets the
standard deviation at which the estimate carries half weight.
"""

for threshold in (0.005, 0.008, 0.010, 0.012, 0.015, 0.020, 0.030):
    blended = blend_by_confidence(surface, std, std_threshold=threshold)
    table = apply_correction(base, blended)
    t = evaluate_table(table, truth, counts)
    a = evaluate_afr(table, test)
    print(f"threshold {threshold:6.3f}  RMSE {t.rmse:6.2f}  visited {t.rmse_visited:6.2f}  "
          f"unvis {t.rmse_unvisited:6.2f}  AFR {a.mean_abs_pct:5.2f}%")

In [ ]:
"""
Map the signed table error of each method across the operating range.
A shared colour scale makes the panels directly comparable.
"""

extent = [DEFAULT_RPM_AXIS[0], DEFAULT_RPM_AXIS[-1], DEFAULT_MAP_AXIS[0], DEFAULT_MAP_AXIS[-1]]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for ax, method in zip(axes.ravel(), SURFACE_METHODS, strict=True):
    error = (results[method] - truth) * 100
    limit = 15
    im = ax.imshow(error.T, origin="lower", aspect="auto", extent=extent,
                   cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_title(f"{method}  (RMSE {evaluate_table(results[method], truth).rmse:.2f})")
    fig.colorbar(im, ax=ax)

fig.suptitle("Table error by fitting method, VE percentage points")
fig.tight_layout()

In [ ]:
"""
Inspect the length scales the Gaussian process learned from the data.
"""
cells = aggregate_to_cells(factors, kept.rpm, kept.map_kpa)
variance = np.where(cells.count > 1, cells.variance, np.median(cells.variance[cells.variance > 0]))
alpha = np.maximum(variance / cells.count, 1e-8)

kernel = ConstantKernel(1.0, (1e-5, 1e5)) * Matern(
    length_scale=[2.0, 2.0], length_scale_bounds=(0.1, 100.0), nu=2.5
) + WhiteKernel(1e-4, (1e-10, 1e-1))

model = GaussianProcessRegressor(kernel=kernel, alpha=alpha, n_restarts_optimizer=2, random_state=0)
model.fit(cells.coords, cells.mean - cells.mean.mean())

rpm_ls, map_ls = model.kernel_.k1.k2.length_scale
print(f"fitted length scale, rpm axis:  {rpm_ls:.2f} cells "
      f"({rpm_ls * np.mean(np.diff(DEFAULT_RPM_AXIS)):.0f} rpm)")
print(f"fitted length scale, load axis: {map_ls:.2f} cells "
      f"({map_ls * np.mean(np.diff(DEFAULT_MAP_AXIS)):.1f} kPa)")
print(f"\nfitted kernel: {model.kernel_}")